# SelectiveNet — Bike Sharing (LSTM)

In [1]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, json, os, sys
from sklearn.metrics import r2_score
sys.path.insert(0, os.path.join('.', '..'))
from shared.data_utils import load_appliance_energy_bike_style

CONFIG = {'method': 'selectivenet', 'lstm_hidden': 64, 'lstm_layers': 1, 'target_coverage': 0.7,
          'lambda_cov': 10.0, 'epochs': 100, 'batch_size': 64, 'lr': 1e-3, 'seeds': [42, 43, 44]}
RESULT_DIR = os.path.join('.', 'results', 'selectivenet')
os.makedirs(RESULT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [2]:
class SelectiveNetLSTM(nn.Module):
    def __init__(self, n_features, lstm_hidden, lstm_layers):
        super().__init__()
        self.lstm = nn.LSTM(n_features, lstm_hidden, lstm_layers, batch_first=True)
        self.pred_head = nn.Linear(lstm_hidden, 1)
        self.sel_head = nn.Linear(lstm_hidden, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        h = lstm_out[:, -1, :]
        return self.pred_head(h), torch.sigmoid(self.sel_head(h))

def selective_loss(y, y_hat, s, target_cov, lam):
    mse = (y - y_hat) ** 2
    cov_loss = lam * F.relu(target_cov - s.mean()) ** 2
    reg_loss = (s * mse).sum() / (s.sum() + 1e-8)
    return reg_loss + cov_loss
print('Model defined.')

Model defined.


In [3]:
def train_one_seed(seed):
    print(f'\n--- Seed {seed} ---')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, (seq_len, n_feat) = \
        load_appliance_energy_bike_style(random_state=seed)
    train_ds = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    model = SelectiveNetLSTM(n_feat, CONFIG['lstm_hidden'], CONFIG['lstm_layers']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    best_state, best_val = None, float('inf')
    for _ in range(CONFIG['epochs']):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            y_hat, s = model(xb)
            loss = selective_loss(yb, y_hat, s, CONFIG['target_coverage'], CONFIG['lambda_cov'])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            y_hat_v, s_v = model(X_val.to(DEVICE))
            vl = selective_loss(y_val.to(DEVICE), y_hat_v, s_v, CONFIG['target_coverage'], CONFIG['lambda_cov']).item()
        if vl < best_val: best_val = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        y_hat, s = model(X_test.to(DEVICE))
    return y_hat.cpu().numpy().squeeze(), s.cpu().numpy().squeeze(), y_test.numpy().squeeze()

for seed in CONFIG['seeds']:
    y_pred, scores, y_true = train_one_seed(seed)
    sd = os.path.join(RESULT_DIR, f'seed_{seed}'); os.makedirs(sd, exist_ok=True)
    np.save(os.path.join(sd, 'test_predictions.npy'), y_pred)
    np.save(os.path.join(sd, 'test_scores.npy'), scores)
    np.save(os.path.join(sd, 'test_labels.npy'), y_true)
    print(f'  R²: {r2_score(y_true, y_pred):.4f}')
print(f'\nDone. Saved to {RESULT_DIR}')


--- Seed 42 ---


  R²: 0.1951

--- Seed 43 ---


  R²: 0.2244

--- Seed 44 ---


  R²: 0.1521

Done. Saved to .\results\selectivenet
